# cropy · 3주차 CNN 실험
순서: 번들 업로드 → 기준 모델 val 평가 → 무 오류 검토 → 대조군/균형 샘플러 이어학습 → val 비교 → 결과 다운로드.

**런타임에서 GPU를 선택하고 위에서부터 실행하세요.** 학습은 비용과 시간이 발생할 수 있습니다.
`cropy_week3_bundle.zip`을 사용합니다. 현재 모델과 기존 Colab 번들의 축소 이미지를 포함합니다.
과거 원본 이미지 test 93.1%와 직접 비교하지 않습니다. 보드 속도 측정도 아닙니다.
기존 모델을 덮어쓰지 않습니다. 결과는 세션별 폴더에 저장합니다. 런타임 종료 전 반드시 다운로드하세요.
test는 기본 실행에서 사용하지 않으며, 실험 선택은 val로만 합니다.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, json, hashlib, datetime, sys, subprocess, shutil, csv, random, runpy
import numpy as np
import torch, torchvision
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
from PIL import Image
import matplotlib.pyplot as plt

assert torch.cuda.is_available(), 'GPU 런타임을 선택한 뒤 다시 실행하세요.'
print(torch.cuda.get_device_name(0), torch.__version__, torchvision.__version__)
uploaded=files.upload()  # cropy_week3_bundle.zip 하나 선택
assert len(uploaded)==1, '번들 ZIP 하나만 선택하세요.'
archive=Path(next(iter(uploaded)))
work=Path('/content')/('cropy_'+datetime.datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
work.mkdir()
with zipfile.ZipFile(archive) as z:
    for name in z.namelist():
        assert (work/name).resolve().is_relative_to(work.resolve()), '잘못된 ZIP 경로'
    z.extractall(work)
del uploaded
data=work/'dataset'
baseline=work/'checkpoints/best_model.pth'
out=work/'results'; out.mkdir()
def sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''): h.update(chunk)
    return h.hexdigest()
manifest=json.loads((work/'manifest.json').read_text())
assert sha(baseline)==manifest['checkpoint_sha256'], '체크포인트 해시 불일치'
assert sha(work/'train.py')==manifest['train_sha256'], '학습 코드 해시 불일치'
ckpt=torch.load(baseline,map_location='cpu',weights_only=True)
size=int(ckpt.get('img_size',224))
classes=ckpt['classes']; arch=ckpt.get('arch','resnet18')
assert ckpt['num_classes']==len(classes)==9
print('모델:',arch,'입력:',size,'클래스:',classes)
print('img_size 누락 시 기존 서비스와 동일하게 224 사용:', 'img_size' not in ckpt)
for split in ['train','val','test']:
    ds=datasets.ImageFolder(data/split)
    assert ds.classes==classes, f'{split} 클래스 순서 불일치'
    print(split,len(ds))
environment={'torch':torch.__version__,'torchvision':torchvision.__version__,
 'gpu':torch.cuda.get_device_name(0),'img_size':size,'arch':arch,
 'baseline_sha256':sha(baseline),'bundle_sha256':sha(archive),
 'dataset_note':manifest['dataset_note']}
(out/'environment.json').write_text(json.dumps(environment,ensure_ascii=False,indent=2))
shutil.copy2(work/'manifest.json',out/'manifest.json')


## 1. 기준 모델 평가
val로 정확도·클래스별 정확도·혼동행렬·전체 예측 CSV를 저장합니다.
원본 모델의 해상도를 유지하며 TTA는 사용하지 않습니다.


In [ ]:
device=torch.device('cuda')
tf=transforms.Compose([transforms.Resize((size,size)),transforms.ToTensor(),
 transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
def assess(checkpoint,label,split='val'):
    state=torch.load(checkpoint,map_location='cpu',weights_only=True)
    assert state['classes']==classes
    assert int(state.get('img_size',224))==size
    model=getattr(models,state.get('arch','resnet18'))(weights=None)
    model.fc=torch.nn.Linear(model.fc.in_features,len(classes))
    model.load_state_dict(state['model_state_dict']); model.to(device).eval()
    ds=datasets.ImageFolder(data/split,transform=tf)
    assert ds.classes==classes
    cm=np.zeros((len(classes),len(classes)),dtype=int); rows=[]; offset=0
    with torch.inference_mode():
        for x,y in DataLoader(ds,batch_size=32,shuffle=False,num_workers=0):
            prob=model(x.to(device)).softmax(1).cpu()
            for j,(actual,pred) in enumerate(zip(y.tolist(),prob.argmax(1).tolist())):
                path=Path(ds.samples[offset+j][0])
                cm[actual,pred]+=1
                rows.append({'path':str(path.relative_to(data)),'actual':actual,
                 'predicted':pred,'score':float(prob[j,pred])})
            offset+=len(y)
    totals=cm.sum(1)
    result={'label':label,'split':split,'checkpoint_sha256':sha(checkpoint),
     'n':int(cm.sum()),'accuracy':float(cm.trace()/cm.sum()),'classes':classes,
     'per_class_accuracy':np.divide(cm.diagonal(),totals).tolist(),
     'confusion':cm.tolist(),'img_size':size,'tta':False}
    (out/f'{label}_{split}.json').write_text(json.dumps(result,ensure_ascii=False,indent=2))
    with open(out/f'{label}_{split}_predictions.csv','w',newline='',encoding='utf-8-sig') as f:
        writer=csv.DictWriter(f,fieldnames=['path','actual','predicted','score'])
        writer.writeheader();writer.writerows(rows)
    fig,ax=plt.subplots(figsize=(8,7)); ax.imshow(cm,cmap='YlGn')
    for i in range(len(classes)):
        for j in range(len(classes)): ax.text(j,i,str(cm[i,j]),ha='center',va='center')
    ax.set(xlabel='Predicted class ID',ylabel='Actual class ID',title=f'{label} / {split}',
           xticks=range(9),yticks=range(9))
    fig.savefig(out/f'{label}_{split}_confusion.png',bbox_inches='tight');plt.show();plt.close(fig)
    print(label,split,round(result['accuracy']*100,2),'%')
    for name,acc,n in zip(classes,result['per_class_accuracy'],totals): print(name,round(acc*100,2),int(n))
    del model;torch.cuda.empty_cache()
    return result,rows
base_result,base_rows=assess(baseline,'baseline')


## 2. 무 두 병해의 오류 확인
이미지는 val 오류만 확인합니다. 표시된 클래스 ID는 앞 셀의 목록을 참고하세요. 원인을 본 뒤 아래 오류 메모를 작성하세요.


In [ ]:
errors=[r for r in base_rows if r['actual'] in [2,3] and r['predicted']!=r['actual']]
print('무 병해 val 오류:',len(errors))
if errors:
    fig,axes=plt.subplots(3,4,figsize=(14,10))
    for ax in axes.flat: ax.axis('off')
    for ax,row in zip(axes.flat,errors[:12]):
        with Image.open(data/row['path']) as im: ax.imshow(im.convert('RGB'))
        ax.set_title(f"true={row['actual']} pred={row['predicted']} score={row['score']:.2f}")
    fig.tight_layout();fig.savefig(out/'radish_val_errors.png');plt.show();plt.close(fig)
error_notes='여기에 관찰 기록: 라벨 의심 / 촬영 부위 / 흐림 / 배경 / 판단 불가'
(out/'error_notes.txt').write_text(error_notes,encoding='utf-8')


## 3. 같은 조건으로 두 번 이어학습
기준 모델과의 비교만으로는 추가 학습 효과와 샘플러 효과를 구분할 수 없습니다.
동일 시작 가중치·해상도·seed·학습률로 **일반 샘플링 대조군 / 균형 샘플링 실험군**을 각각 학습합니다.
두 실행 모두 baseline에서 시작합니다. val 전체 정확도가 시작 모델을 넘을 때만 새 파일이 생깁니다.
개선이 없으면 baseline 유지로 기록합니다. 단일 seed 실험이므로 일반화된 개선을 단정하지 않습니다.


In [ ]:
SEED=42  # runner에도 동일하게 고정: 이번 실험에서는 변경하지 않음
EPOCHS=12
BATCH_SIZE=32  # 메모리 부족 시 두 실험 모두 같은 값으로 줄여 새로 실행
LR=2e-5
runner=work/'seeded_train.py'
runner.write_text("import random, numpy as np, torch, runpy, sys\n"
 "random.seed(42); np.random.seed(42); torch.manual_seed(42); torch.cuda.manual_seed_all(42)\n"
 "torch.backends.cudnn.benchmark=False\n"
 "torch.backends.cudnn.deterministic=True\n"
 "runpy.run_path(sys.argv[0].replace('seeded_train.py','train.py'),run_name='__main__')\n")
session=out/('training_'+datetime.datetime.now().strftime('%H%M%S_%f'));session.mkdir()
config={'seed':SEED,'epochs':EPOCHS,'batch_size':BATCH_SIZE,'lr':LR,'img_size':size,
 'selection':'strict improvement over starting overall val accuracy','start_sha256':sha(baseline)}
(session/'config.json').write_text(json.dumps(config,indent=2))
candidates={}
for label,balanced in [('control',False),('balanced',True)]:
    dest=session/f'{label}.pth'
    cmd=[sys.executable,str(runner),'--data-dir',str(data),'--arch',arch,
     '--resume',str(baseline),'--img-size',str(size),'--lr',str(LR),'--epochs',str(EPOCHS),
     '--batch-size',str(BATCH_SIZE),'--num-workers','0','--output',str(dest)]
    if balanced: cmd.append('--balanced-sampler')
    with open(session/f'{label}.log','w') as log:
        p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in p.stdout: print(line,end='');log.write(line);log.flush()
        if p.wait()!=0: raise RuntimeError(f'{label} 학습 실패. 로그 확인 후 재실행하세요.')
    candidates[label]=dest if dest.exists() else baseline
    print(label, '새 후보 저장' if dest.exists() else 'val 개선 없음: 기준 모델 유지')
assert sha(baseline)==environment['baseline_sha256']


## 4. val 비교와 후보 결정
무 두 클래스의 변화와 전체 정확도를 함께 확인합니다. 개선이 없다면 기존 모델을 유지하세요. 아래 표는 화면에만 표시되는 것이 아니라 CSV로도 저장됩니다.


In [ ]:
results={'baseline':base_result}
for label,path in candidates.items(): results[label],_=assess(path,label)
comparison=[]
for label,r in results.items():
    comparison.append({'model':label,'overall':r['accuracy'],
     'radish_black_spot':r['per_class_accuracy'][2],
     'radish_downy_mildew':r['per_class_accuracy'][3],
     'checkpoint_sha256':r['checkpoint_sha256']})
import pandas as pd
table=pd.DataFrame(comparison);display(table)
table.to_csv(out/'comparison_val.csv',index=False,encoding='utf-8-sig')


## 5. 최종 test 평가 — 기본값은 실행 안 함
val 비교와 검토를 끝내고 후보를 확정한 뒤에만 실행하세요. test를 보고 다시 후보를 고르는 용도로 사용하지 마세요.
같은 축소 데이터의 baseline과 최종 후보를 평가합니다. 보드/원본 데이터 성능과는 구분합니다.


In [ ]:
FINALIZE_TEST=False
SELECTED='baseline'  # 검토 후 baseline / control / balanced 중 직접 지정
SELECTION_REASON=''  # val 비교에 근거한 선택 이유
if FINALIZE_TEST:
    assert SELECTED in results and SELECTION_REASON.strip(), '후보와 선택 근거를 입력하세요.'
    marker=out/'test_selection.json'
    assert not marker.exists(), '이 실행에서 test 평가를 이미 확정했습니다.'
    selected_path=baseline if SELECTED=='baseline' else candidates[SELECTED]
    marker.write_text(json.dumps({'selected':SELECTED,'reason':SELECTION_REASON,
     'sha256':sha(selected_path)},ensure_ascii=False,indent=2))
    assess(baseline,'baseline','test')
    if sha(selected_path)!=sha(baseline): assess(selected_path,'selected','test')
else:
    print('test 미실행. val 결과를 검토한 뒤 후보를 확정하세요.')


## 6. 결과 다운로드
학습 로그·환경·평가표·혼동행렬·저장된 실험 체크포인트를 함께 받습니다. 원본 baseline은 입력 번들에 보관되어 있습니다. 로컬 모델을 자동 교체하지 않습니다.


In [ ]:
assert sha(baseline)==environment['baseline_sha256']
archive_path=shutil.make_archive(str(work/'cropy_week3_results'),'zip',out)
files.download(archive_path)
print('다운로드 후 comparison_val.csv와 training 폴더의 로그를 확인하세요.')
